In [6]:
import os
import numpy as np
import pandas as pd
import json
import shutil
import requests
from tqdm import tqdm

In [7]:
with open("./def_templates/exomol.json", "r", encoding="utf-8") as f:
    exomol_template = json.load(f)

def empty_dict(d):
    for key, value in d.items():
        if isinstance(value, list):
            d[key] = []
        elif isinstance(value, dict):
            keys = list(value.keys())
            d[key] = empty_dict(d[key])
        else:
            d[key] = None
    return d

exomol_template = empty_dict(exomol_template)

with open("./def_templates/exomol.json", "w", encoding="utf-8") as f:
    json.dump(exomol_template, f, indent=4)

In [11]:
db_url = "http://exomol.com/db/"
mol = "AlH"
iso_slug = "27Al-1H"
ds_name = "AloHa"

url = db_url + f"{mol}/{iso_slug}/{ds_name}/{iso_slug}__{ds_name}"
exts = [".pf"]
path = f"./samples/Dozen/{iso_slug}__{ds_name}"
for ext in exts:
    rurl = url + ext
    dest_path = path + ext
    response = requests.get(rurl, stream=True)
    response.raise_for_status()
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    with open(dest_path, 'wb') as f:
        total_size = int(response.headers.get('content-length', 0))
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Downloading {os.path.basename(dest_path)}") as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))